# baseline v4 - Gemma 4 31B-it LoRA

이 베이스라인 코드는 `google/gemma-4-31B-it` 기준의 **QLoRA 파인튜닝** 버전입니다.

주요 변경점
- `Qwen/Qwen2.5-VL-3B-Instruct` → `google/gemma-4-31B-it`
- **비추론 모드**(`enable_thinking=False`) 고정
- **A100 80GB** 기준 LoRA/QLoRA 학습 세팅으로 상향
- **고정 valid set + epoch별 랜덤 train subset 샘플링**
- **epoch마다 체크포인트 저장**
- **체크포인트 기준 resume / inference 로드 가능**
- **overfitting 방지용 early stopping 포함**

> 참고: `google/gemma-4-31B`는 base weights이고, 현재 노트북은 채팅/지시형 VLM 파인튜닝에 맞게 `google/gemma-4-31B-it`를 사용합니다.


# 환경 준비

개발 환경에 필요한 라이브러리를 설치합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작
- Gemma 사용 전 Hugging Face에서 **Gemma 라이선스 동의**가 필요할 수 있습니다.
- Gemma 4는 최신 `transformers` / `peft`가 필요하므로 git main 기준으로 설치합니다.


In [1]:
!pip -q install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install --upgrade git+https://github.com/huggingface/transformers.git
!pip -q install --upgrade git+https://github.com/huggingface/peft.git
!pip -q install --upgrade accelerate bitsandbytes datasets pillow pandas huggingface_hub sentencepiece protobuf


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 147.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflict

In [1]:
import torch, transformers, peft, bitsandbytes as bnb
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)
print("BitsAndBytes version:", bnb.__version__)


Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002
Transformers version: 5.5.0.dev0
PEFT version: 0.18.2.dev0
BitsAndBytes version: 0.49.2


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 Colab에서 구글 드라이브를 마운트하여 사용합니다.


In [2]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# 압축 해제
!unzip "/content/drive/My Drive/2026-ssafy-ai-15-2.zip" -d "/content/"


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정


In [4]:
import os, re, gc, math, json, random, shutil
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"
Image.MAX_IMAGE_PIXELS = None

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# -----------------------------
# 기본 설정
# -----------------------------
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 실제 사용 모델:
# - google/gemma-4-31B     : base
# - google/gemma-4-31B-it  : instruction-tuned
MODEL_ID = "google/gemma-4-31B-it"

# Gemma 4 reasoning 비활성화
ENABLE_THINKING = False

# 학습 경로 / 재시작 설정
RUN_DIR = "/content/gemma4_31b_it_mcqa_lora"
RESUME_FROM_CHECKPOINT = None
# 예: RESUME_FROM_CHECKPOINT = "/content/gemma4_31b_it_mcqa_lora/checkpoint-epoch-03"

# RESUME_FROM_CHECKPOINT가 있으면 자동으로 해당 run 폴더를 사용
if RESUME_FROM_CHECKPOINT:
    RUN_DIR = str(Path(RESUME_FROM_CHECKPOINT).resolve().parent)

# 데이터 분리 / epoch 샘플링
VALID_RATIO = 0.10
SUBSAMPLE_TRAIN_PER_EPOCH = 400   # None 또는 0이면 전체 train pool 사용
NUM_EPOCHS = 10

# 학습 하이퍼파라미터 (A100 80GB 기준 안정성 우선)
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 0.3

# LoRA 설정
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.10

# Gemma 4 vision tower LoRA는 현 시점 PEFT 이슈가 있어 기본은 False 권장
# True로 켜면 아래 모델 로드 셀에서 monkey patch 후 전체 q/k/v/o + mlp까지 LoRA 적용 가능
ENABLE_VISION_TOWER_LORA = False

# 기타
MAX_NEW_TOKENS = 4
NUM_WORKERS = 0
ATTN_IMPLEMENTATION = "eager"  # 안정성 우선. flash_attention_2 직접 설치 시 변경 가능

EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-4

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

os.makedirs(RUN_DIR, exist_ok=True)

# -----------------------------
# 데이터 로드
# -----------------------------
raw_train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

train_pool_path = os.path.join(RUN_DIR, "train_pool_fixed.csv")
valid_path = os.path.join(RUN_DIR, "valid_fixed.csv")

if os.path.exists(train_pool_path) and os.path.exists(valid_path):
    train_pool_df = pd.read_csv(train_pool_path)
    valid_df = pd.read_csv(valid_path)
    print("고정 split 불러옴:", RUN_DIR)
else:
    shuffled_df = raw_train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    valid_size = max(1, int(len(shuffled_df) * VALID_RATIO))
    valid_df = shuffled_df.iloc[:valid_size].reset_index(drop=True)
    train_pool_df = shuffled_df.iloc[valid_size:].reset_index(drop=True)

    train_pool_df.to_csv(train_pool_path, index=False)
    valid_df.to_csv(valid_path, index=False)
    print("고정 split 저장:", RUN_DIR)

samples_per_epoch = len(train_pool_df) if not SUBSAMPLE_TRAIN_PER_EPOCH else min(SUBSAMPLE_TRAIN_PER_EPOCH, len(train_pool_df))
optimizer_steps_per_epoch = math.ceil(samples_per_epoch / PER_DEVICE_BATCH_SIZE / GRAD_ACCUM)
total_optimizer_steps = max(1, optimizer_steps_per_epoch * NUM_EPOCHS)

print(f"Device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("MODEL_ID:", MODEL_ID)
print("USE_BF16:", USE_BF16)
print("train_pool size:", len(train_pool_df))
print("valid size:", len(valid_df))
print("samples_per_epoch:", samples_per_epoch)
print("optimizer_steps_per_epoch:", optimizer_steps_per_epoch)
print("total_optimizer_steps:", total_optimizer_steps)


고정 split 저장: /content/gemma4_31b_it_mcqa_lora
Device available: cuda
MODEL_ID: google/gemma-4-31B-it
USE_BF16: True
train_pool size: 4566
valid size: 507
samples_per_epoch: 400
optimizer_steps_per_epoch: 100
total_optimizer_steps: 1000


# 모델, Processor

- Gemma 4는 공식 예시상 `enable_thinking=False`로 **비추론 모드** 사용 가능
- 31B 모델은 Dense 구조이며 text + image를 지원합니다
- 현재 노트북은 **QLoRA(4bit) + LoRA** 조합으로 로드합니다
- 체크포인트에서 재시작할 수 있도록 adapter / optimizer / scheduler 상태도 같이 저장합니다


In [5]:
# -----------------------------
# Gemma 4 vision tower LoRA용 optional patch
# -----------------------------
def maybe_patch_gemma4_clippable_linear():
    """
    Gemma 4 vision/audio tower의 Gemma4ClippableLinear는 현 시점 일부 PEFT 버전에서
    LoRA 삽입 대상일 때 에러가 날 수 있어, 필요 시 monkey patch로 우회합니다.
    기본값은 ENABLE_VISION_TOWER_LORA=False 이므로 보통은 실행되지 않습니다.
    """
    import torch.nn as nn
    from transformers.models.gemma4 import modeling_gemma4

    class PatchedClippableLinear(nn.Linear):
        def __init__(self, config, in_features, out_features):
            nn.Linear.__init__(self, in_features, out_features, bias=False)
            self.use_clipped_linears = getattr(config, "use_clipped_linears", False)
            if self.use_clipped_linears:
                self.register_buffer("input_min", torch.tensor(-float("inf")))
                self.register_buffer("input_max", torch.tensor(float("inf")))
                self.register_buffer("output_min", torch.tensor(-float("inf")))
                self.register_buffer("output_max", torch.tensor(float("inf")))

        def forward(self, x):
            if self.use_clipped_linears:
                x = torch.clamp(x, self.input_min, self.input_max)
            out = nn.Linear.forward(self, x)
            if self.use_clipped_linears:
                out = torch.clamp(out, self.output_min, self.output_max)
            return out

    modeling_gemma4.Gemma4ClippableLinear = PatchedClippableLinear
    print("Gemma4ClippableLinear patch applied.")

if ENABLE_VISION_TOWER_LORA:
    maybe_patch_gemma4_clippable_linear()

# -----------------------------
# Processor / Quantization
# -----------------------------
processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=TORCH_DTYPE,
    bnb_4bit_quant_storage=TORCH_DTYPE,
)

base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    attn_implementation=ATTN_IMPLEMENTATION,
    low_cpu_mem_usage=True,
)

base_model.config.use_cache = False
if hasattr(base_model.config, "text_config"):
    base_model.config.text_config.use_cache = False

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# -----------------------------
# LoRA target modules
# -----------------------------
if ENABLE_VISION_TOWER_LORA:
    # vision tower까지 함께 LoRA 적용
    lora_target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embedding_projection",
    ]
else:
    # 안정성 우선:
    # - language_model의 attention / MLP
    # - image -> text projection(embed_vision.embedding_projection)
    lora_target_modules = r".*(model\.language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))|model\.embed_vision\.embedding_projection)$"

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=lora_target_modules,
    task_type="CAUSAL_LM",
)

resume_state = None
start_epoch = 1
best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

if RESUME_FROM_CHECKPOINT:
    model = PeftModel.from_pretrained(
        base_model,
        RESUME_FROM_CHECKPOINT,
        is_trainable=True,
    )
    processor = AutoProcessor.from_pretrained(RESUME_FROM_CHECKPOINT)

    trainer_state_path = os.path.join(RESUME_FROM_CHECKPOINT, "trainer_state.pt")
    if os.path.exists(trainer_state_path):
        resume_state = torch.load(trainer_state_path, map_location="cpu")
        start_epoch = int(resume_state["epoch"]) + 1
        best_val_loss = float(resume_state.get("best_val_loss", float("inf")))
        best_epoch = int(resume_state.get("best_epoch", 0))
        patience_counter = int(resume_state.get("patience_counter", 0))
    print("Resume from:", RESUME_FROM_CHECKPOINT)
else:
    model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

target_device = next((p.device for p in model.parameters() if p.requires_grad), next(model.parameters()).device)
print("Target device:", target_device)
print("start_epoch:", start_epoch)
print("best_val_loss:", best_val_loss)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

trainable params: 122,533,888 || all params: 31,395,620,400 || trainable%: 0.3903
Target device: cuda:0
start_epoch: 1
best_val_loss: inf


# 프롬프트 템플릿

Gemma 4 31B-it는 비추론 모드에서도 내부적으로 빈 thought channel을 사용할 수 있으므로,
반드시 `processor.apply_chat_template(..., enable_thinking=False)`를 통해 템플릿을 생성합니다.


In [6]:
SYSTEM_INSTRUCT = (
    "You are a visual multiple-choice question answering assistant. "
    "Thinking mode is disabled. "
    "Do not reveal reasoning. "
    "Return exactly one lowercase letter among a, b, c, or d. "
    "No explanation."
)

def resolve_image_path(path_str: str) -> str:
    path_str = str(path_str)
    if os.path.exists(path_str):
        return path_str
    alt = os.path.join("/content", path_str.lstrip("./"))
    if os.path.exists(alt):
        return alt
    return path_str

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def build_messages_from_row(row, include_answer: bool = False):
    img_path = resolve_image_path(row["path"])
    img = Image.open(img_path).convert("RGB")

    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]),
        str(row["b"]),
        str(row["c"]),
        str(row["d"]),
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text},
        ]},
    ]

    if include_answer:
        gold = str(row["answer"]).strip().lower()
        messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

    return messages, img


# Custom Dataset, Collator

업그레이드 포인트
- **completion-only loss** 적용  
  → 프롬프트 전체를 맞히는 대신 **정답 토큰만** loss에 반영
- **Gemma 4 비추론 템플릿** 적용  
  → `enable_thinking=False`
- **image token / padding token masking** 적용


In [7]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train: bool = True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_messages, img = build_messages_from_row(row, include_answer=False)

        item = {
            "prompt_messages": prompt_messages,
            "image": img,
            "row_id": row["id"] if "id" in row else idx,
        }

        if self.train:
            full_messages, _ = build_messages_from_row(row, include_answer=True)
            item["messages"] = full_messages
            item["answer"] = str(row["answer"]).strip().lower()

        return item

@dataclass
class Gemma4VQACollator:
    processor: Any
    train: bool = True
    enable_thinking: bool = False

    def _get_mask_token_ids(self):
        tokenizer = self.processor.tokenizer
        candidate_ids = set()

        for attr in ["pad_token_id", "image_token_id", "audio_token_id", "video_token_id"]:
            tok_id = getattr(tokenizer, attr, None)
            if tok_id is not None and tok_id >= 0:
                candidate_ids.add(tok_id)

        for attr in ["boi_token", "eoi_token", "boa_token", "eoa_token"]:
            tok = getattr(tokenizer, attr, None)
            if tok is None:
                continue
            tok_id = tokenizer.convert_tokens_to_ids(tok)
            if tok_id is not None and tok_id >= 0:
                candidate_ids.add(tok_id)

        return candidate_ids

    def __call__(self, batch):
        images = [sample["image"] for sample in batch]

        prompt_texts = [
            self.processor.apply_chat_template(
                sample["prompt_messages"],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=self.enable_thinking,
            ).strip()
            for sample in batch
        ]

        if not self.train:
            return self.processor(
                text=prompt_texts,
                images=images,
                padding=True,
                return_tensors="pt",
            )

        full_texts = [
            self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=self.enable_thinking,
            ).strip()
            for sample in batch
        ]

        full_batch = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )
        prompt_batch = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = full_batch["input_ids"].clone()

        # padding mask
        labels[full_batch["attention_mask"] == 0] = -100

        # prompt 부분은 loss 제외 -> assistant completion만 학습
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1).tolist()
        for i, prompt_len in enumerate(prompt_lengths):
            labels[i, :int(prompt_len)] = -100

        # multimodal special tokens / padding은 loss 제외
        for tok_id in self._get_mask_token_ids():
            labels[labels == tok_id] = -100

        full_batch["labels"] = labels
        return full_batch


# DataLoader

- valid set은 고정
- train set은 `train_pool`에서 **epoch마다 랜덤 샘플링**
- 예: 전체 5000개 중 epoch당 200개만 사용하면, 매 epoch마다 200개를 새로 랜덤 추출


In [8]:
valid_ds = VQAMCDataset(valid_df, train=True)
valid_loader = DataLoader(
    valid_ds,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=False,
    collate_fn=Gemma4VQACollator(processor, train=True, enable_thinking=ENABLE_THINKING),
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

def sample_train_epoch_df(epoch: int) -> pd.DataFrame:
    if not SUBSAMPLE_TRAIN_PER_EPOCH or SUBSAMPLE_TRAIN_PER_EPOCH >= len(train_pool_df):
        return train_pool_df.sample(frac=1.0, random_state=SEED + epoch).reset_index(drop=True)

    return train_pool_df.sample(
        n=SUBSAMPLE_TRAIN_PER_EPOCH,
        random_state=SEED + epoch,
        replace=False,
    ).reset_index(drop=True)

def build_train_loader_for_epoch(epoch: int):
    epoch_df = sample_train_epoch_df(epoch)
    epoch_ds = VQAMCDataset(epoch_df, train=True)

    g = torch.Generator()
    g.manual_seed(SEED + epoch)

    train_loader = DataLoader(
        epoch_ds,
        batch_size=PER_DEVICE_BATCH_SIZE,
        shuffle=True,
        generator=g,
        collate_fn=Gemma4VQACollator(processor, train=True, enable_thinking=ENABLE_THINKING),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return epoch_df, train_loader

print("valid_loader batches:", len(valid_loader))
epoch_preview_df, epoch_preview_loader = build_train_loader_for_epoch(start_epoch)
print("train samples for first runnable epoch:", len(epoch_preview_df))
print("train_loader batches for first runnable epoch:", len(epoch_preview_loader))
del epoch_preview_df, epoch_preview_loader
gc.collect()


valid_loader batches: 507
train samples for first runnable epoch: 400
train_loader batches for first runnable epoch: 400


60

# fine-tuning

기능
- epoch별 랜덤 train subset 샘플링
- valid loss 기반 early stopping
- epoch별 checkpoint 저장
- checkpoint 기준 resume 지원
- best adapter / final adapter 별도 저장


In [ ]:
def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device):
    moved = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            moved[k] = v.to(device)
        else:
            moved[k] = v
    return moved

def save_text(path: str, text: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def read_text(path: str, default: str = "") -> str:
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def save_checkpoint(
    epoch: int,
    model,
    processor,
    optimizer,
    scheduler,
    scaler,
    best_val_loss: float,
    best_epoch: int,
    patience_counter: int,
    history: List[Dict[str, Any]],
):
    ckpt_dir = os.path.join(RUN_DIR, f"checkpoint-epoch-{epoch:02d}")
    os.makedirs(ckpt_dir, exist_ok=True)

    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)

    trainer_state = {
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "patience_counter": patience_counter,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "config": {
            "MODEL_ID": MODEL_ID,
            "ENABLE_THINKING": ENABLE_THINKING,
            "VALID_RATIO": VALID_RATIO,
            "SUBSAMPLE_TRAIN_PER_EPOCH": SUBSAMPLE_TRAIN_PER_EPOCH,
            "NUM_EPOCHS": NUM_EPOCHS,
            "PER_DEVICE_BATCH_SIZE": PER_DEVICE_BATCH_SIZE,
            "GRAD_ACCUM": GRAD_ACCUM,
            "LEARNING_RATE": LEARNING_RATE,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "WARMUP_RATIO": WARMUP_RATIO,
            "MAX_GRAD_NORM": MAX_GRAD_NORM,
            "LORA_R": LORA_R,
            "LORA_ALPHA": LORA_ALPHA,
            "LORA_DROPOUT": LORA_DROPOUT,
            "ENABLE_VISION_TOWER_LORA": ENABLE_VISION_TOWER_LORA,
            "EARLY_STOP_PATIENCE": EARLY_STOP_PATIENCE,
            "EARLY_STOP_MIN_DELTA": EARLY_STOP_MIN_DELTA,
            "ATTN_IMPLEMENTATION": ATTN_IMPLEMENTATION,
            "TORCH_DTYPE": str(TORCH_DTYPE),
            "SEED": SEED,
        },
    }
    torch.save(trainer_state, os.path.join(ckpt_dir, "trainer_state.pt"))

    save_text(os.path.join(RUN_DIR, "latest_checkpoint.txt"), ckpt_dir)
    pd.DataFrame(history).to_csv(os.path.join(RUN_DIR, "train_history.csv"), index=False)
    return ckpt_dir

# Optimizer / scheduler / scaler
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_optimizer_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=(torch.cuda.is_available() and TORCH_DTYPE == torch.float16))

history = []
if resume_state is not None:
    if resume_state.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(resume_state["optimizer_state_dict"])
    if resume_state.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(resume_state["scheduler_state_dict"])
    if resume_state.get("scaler_state_dict") is not None and scaler.is_enabled():
        scaler.load_state_dict(resume_state["scaler_state_dict"])
    history = resume_state.get("history", [])

best_adapter_dir = os.path.join(RUN_DIR, "best_adapter")
final_adapter_dir = os.path.join(RUN_DIR, "final_adapter")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    # -----------------------------
    # epoch별 train subset 랜덤 샘플링
    # -----------------------------
    epoch_train_df, train_loader = build_train_loader_for_epoch(epoch)
    epoch_train_df.to_csv(os.path.join(RUN_DIR, f"sampled_train_epoch_{epoch:02d}.csv"), index=False)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_batches = 0
    grad_accum_counter = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(train_bar, start=1):
        batch = move_batch_to_device(batch, target_device)

        with torch.autocast(device_type="cuda", dtype=TORCH_DTYPE, enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss

        loss_to_backward = loss / GRAD_ACCUM

        if scaler.is_enabled():
            scaler.scale(loss_to_backward).backward()
        else:
            loss_to_backward.backward()

        train_loss_sum += float(loss.item())
        train_batches += 1
        grad_accum_counter += 1

        should_step = (grad_accum_counter == GRAD_ACCUM) or (step == len(train_loader))
        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=MAX_GRAD_NORM,
            )

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            grad_accum_counter = 0

        train_bar.set_postfix({"train_loss": f"{train_loss_sum / max(train_batches, 1):.4f}"})

    avg_train_loss = train_loss_sum / max(train_batches, 1)

    # -----------------------------
    # validation
    # -----------------------------
    model.eval()
    valid_loss_sum = 0.0
    valid_batches = 0

    with torch.no_grad():
        valid_bar = tqdm(valid_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [valid]", unit="batch")
        for batch in valid_bar:
            batch = move_batch_to_device(batch, target_device)

            with torch.autocast(device_type="cuda", dtype=TORCH_DTYPE, enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs.loss

            valid_loss_sum += float(loss.item())
            valid_batches += 1
            valid_bar.set_postfix({"valid_loss": f"{valid_loss_sum / max(valid_batches, 1):.4f}"})

    avg_valid_loss = valid_loss_sum / max(valid_batches, 1)

    improved = avg_valid_loss < (best_val_loss - EARLY_STOP_MIN_DELTA)
    if improved:
        best_val_loss = avg_valid_loss
        best_epoch = epoch
        patience_counter = 0

        os.makedirs(best_adapter_dir, exist_ok=True)
        model.save_pretrained(best_adapter_dir)
        processor.save_pretrained(best_adapter_dir)
        save_text(os.path.join(RUN_DIR, "best_checkpoint.txt"), os.path.join(RUN_DIR, f"checkpoint-epoch-{epoch:02d}"))
    else:
        patience_counter += 1

    history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "valid_loss": avg_valid_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "patience_counter": patience_counter,
        "sampled_train_size": len(epoch_train_df),
    })

    ckpt_dir = save_checkpoint(
        epoch=epoch,
        model=model,
        processor=processor,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        best_val_loss=best_val_loss,
        best_epoch=best_epoch,
        patience_counter=patience_counter,
        history=history,
    )

    print(
        f"[Epoch {epoch}] "
        f"train_loss={avg_train_loss:.4f} | "
        f"valid_loss={avg_valid_loss:.4f} | "
        f"best_epoch={best_epoch} | "
        f"best_val_loss={best_val_loss:.4f} | "
        f"checkpoint={ckpt_dir}"
    )

    del epoch_train_df, train_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

# final adapter 저장
os.makedirs(final_adapter_dir, exist_ok=True)
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)

print("Best adapter:", best_adapter_dir)
print("Final adapter:", final_adapter_dir)
print("Latest checkpoint:", read_text(os.path.join(RUN_DIR, "latest_checkpoint.txt")))
print("Best checkpoint:", read_text(os.path.join(RUN_DIR, "best_checkpoint.txt")))


# inference

- 기본적으로 `best_adapter`를 로드해서 추론합니다
- 다른 체크포인트를 쓰고 싶으면 `ADAPTER_FOR_INFERENCE`만 바꾸면 됩니다
- 체크포인트 폴더(`checkpoint-epoch-xx`)도 그대로 로드 가능합니다


In [9]:
def move_batch_to_device(batch, device):
    moved = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            moved[k] = v.to(device)
        else:
            moved[k] = v
    return moved

def extract_choice(text: str) -> str:
    text = str(text).strip().lower()
    matches = re.findall(r"\b([abcd])\b", text)
    if matches:
        return matches[-1]
    for ch in ["a", "b", "c", "d"]:
        if ch in text:
            return ch
    return "a"

def safe_parse_response(processor, response: str) -> str:
    try:
        parsed = processor.parse_response(response)
        if isinstance(parsed, dict):
            return str(parsed.get("content", response))
    except Exception:
        pass
    return response

def get_stop_token_ids(tokenizer):
    stop_ids = []
    eos_token_id = getattr(tokenizer, "eos_token_id", None)

    if isinstance(eos_token_id, list):
        stop_ids.extend(eos_token_id)
    elif eos_token_id is not None:
        stop_ids.append(eos_token_id)

    end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if end_of_turn_id is not None and end_of_turn_id >= 0:
        stop_ids.append(end_of_turn_id)

    return sorted(set(stop_ids))

ADAPTER_FOR_INFERENCE = None
# ADAPTER_FOR_INFERENCE = "/content/gemma4_31b_it_mcqa_lora/checkpoint-epoch-04"

try:
    del model
    del base_model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if ADAPTER_FOR_INFERENCE is None:
    infer_processor = AutoProcessor.from_pretrained(MODEL_ID)
else:
    infer_processor = AutoProcessor.from_pretrained(ADAPTER_FOR_INFERENCE)

infer_base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    attn_implementation=ATTN_IMPLEMENTATION,
    low_cpu_mem_usage=True,
)

if ADAPTER_FOR_INFERENCE is None:
    infer_model = infer_base_model
    loaded_name = f"{MODEL_ID} (no adapter)"
else:
    infer_model = PeftModel.from_pretrained(infer_base_model, ADAPTER_FOR_INFERENCE)
    loaded_name = ADAPTER_FOR_INFERENCE

infer_model.eval()
infer_target_device = next(infer_model.parameters()).device
stop_token_ids = get_stop_token_ids(infer_processor.tokenizer)

preds = []
output_text = None

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    messages, img = build_messages_from_row(row, include_answer=False)

    text = infer_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    ).strip()

    inputs = infer_processor(
        text=[text],
        images=[img],
        padding=True,
        return_tensors="pt",
    )
    inputs = move_batch_to_device(inputs, infer_target_device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        generated = infer_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=stop_token_ids,
            pad_token_id=infer_processor.tokenizer.pad_token_id,
        )

    response = infer_processor.decode(generated[0][input_len:], skip_special_tokens=False)
    output_text = safe_parse_response(infer_processor, response)
    preds.append(extract_choice(output_text))

submission_path = os.path.join(RUN_DIR, "submission.csv")
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Model used:", loaded_name)

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Inference:   0%|          | 0/5074 [00:00<?, ?sample/s]

NameError: name 'move_batch_to_device' is not defined

In [ ]:
print("RUN_DIR:", RUN_DIR)
print("Best checkpoint:", read_text(os.path.join(RUN_DIR, "best_checkpoint.txt")))
print("Latest checkpoint:", read_text(os.path.join(RUN_DIR, "latest_checkpoint.txt")))
print("Best adapter dir:", best_adapter_dir)
print("Final adapter dir:", final_adapter_dir)
print("Sample output:", output_text)
